# ゼロから作るショアのアルゴリズム 第3回: 制御剰余乗算

[第1回](ja/401_shor_scratch_adder.ipynb)では通常の加算器を、[第2回](ja/402_shor_scratch_modulo_adder.ipynb)では剰余加算 $a+b \bmod N$ を作りました。今回は**制御剰余乗算**を実装します。固定された古典定数 $a$ が与えられたとき、次を計算します。

$$
\ket{\text{ctrl}}\ket{x}\ket{0} \longrightarrow \ket{\text{ctrl}}\ket{x}\ket{\text{ctrl}\cdot(a x \bmod N)}
$$

つまり、$x$ に定数 $a$ を掛けて $N$ で割った余りを計算しますが、これは追加の制御量子ビットが $1$ のときだけ行われ、制御が $0$ のときは結果レジスタは(依然として $\ket 0$ のまま)変化しません。これはまさに、次回作る**べき剰余**を構築するために繰り返し使う演算です: $a^j \bmod N$ は、$j$ の各ビットで制御された $a, a^2, a^4, a^8, \ldots$ による制御乗算の連鎖です。

参考文献: V. Vedral, A. Barenco, A. Ekert, ["Quantum Networks for Elementary Arithmetic Operations"](https://arxiv.org/abs/quant-ph/9511018) (1995)

## アイデア: 乗算は条件付き足し算の繰り返し

2進数の筆算による乗算とは: $x$ の各ビット $x_i$ について、$x_i=1$ ならば $a \cdot 2^i$ を合計に加える、というものです。この処理を(最後に一度だけでなく)**毎回 $N$ で割った余りを取りながら**行うことで、すべての中間値を範囲内に収めます。これは重要です。なぜなら、加算器のレジスタは $N$ 未満の値しか保持できるように設計されていないからです。

したがって回路は次のようになります: $i = 0, \ldots, n-1$ について、第2回の剰余加算器を使って*定数* $(a \cdot 2^i) \bmod N$ を結果レジスタ $b$ に加えます — ただし、重ね合わせの中で**乗算器自身の制御量子ビット**と$x_i$の**両方**が $1$ になっている枝だけに対して行います。

この「2つの追加量子ビットが両方とも1のときだけ」という要求は、剰余加算器の中のすべてのゲート — 第2回ですでに2つ・3つの量子ビットで制御されていたゲートも含めて — に、さらに2つ制御を積み増す必要があることを意味します。第2回の使い切りのアンシラのテクニックではなく、既存の回路に任意の個数の制御を追加する一般的な方法が必要です。

In [1]:
!pip install git+https://github.com/blueqat/blueqatSDK

/Users/yuichirominato/.pyenv/shims/pip: line 8: /opt/homebrew/opt/pyenv/bin/pyenv: No such file or directory


## 一般的な多制御Xゲート

$k$個の量子ビットで制御された($k \ge 3$)$X$ゲートを実装する標準的な手法では、$k-2$個の補助(「アンシラ」)量子ビットを使います: 最初の2つの制御を `CCX` でアンシラにANDし、そのアンシラと次の制御をさらに別のアンシラにAND…というように、すべての$k$個の制御を1個の量子ビットへと連鎖させ、最終的にそれが本当のターゲットを制御します。その後、同じ `CCX` の連鎖を逆順に実行すれば、すべてのアンシラが$\ket 0$ に戻ります(`CCX` は自己逆元なので、「元に戻す」とは「逆順にもう一度実行する」ことに他なりません)。

In [2]:
from blueqat import Circuit

def mcx(controls, target, ancillas):
    """X on `target`, controlled by ALL qubits in `controls`. Uses `ancillas` as scratch
    (len(ancillas) >= len(controls) - 2), restored to |0> on exit."""
    circ = Circuit()
    n = len(controls)
    if n == 0:
        circ.x[target]
        return circ
    if n == 1:
        circ.cx[controls[0], target]
        return circ
    if n == 2:
        circ.ccx[controls[0], controls[1], target]
        return circ
    chain = []
    circ.ccx[controls[0], controls[1], ancillas[0]]
    chain.append(ancillas[0])
    for i in range(2, n - 1):
        circ.ccx[chain[-1], controls[i], ancillas[len(chain)]]
        chain.append(ancillas[len(chain)])
    circ.ccx[chain[-1], controls[-1], target]
    for i in range(len(chain) - 1, 0, -1):
        circ.ccx[chain[i - 1], controls[i + 1], chain[i]]
    circ.ccx[controls[0], controls[1], chain[0]]
    return circ

In [3]:
# Sanity check: a 4-controlled-X should fire only when all 4 controls are 1, and leave ancillas clean.
import itertools
controls, target, ancillas = [0, 1, 2, 3], 4, [5, 6]
for bits in itertools.product([0, 1], repeat=4):
    c = Circuit(7)
    for i, bit in enumerate(bits):
        if bit:
            c.x[i]
    c += mcx(controls, target, ancillas)
    state = c.m[:].run(shots=1).most_common(1)[0][0]
    got_target = int(state[7 - 1 - target])
    got_ancillas = [int(state[7 - 1 - a]) for a in ancillas]
    expected = 1 if all(bits) else 0
    status = "OK" if got_target == expected and not any(got_ancillas) else "MISMATCH"
    if status != "OK":
        print(bits, "->", got_target, "expected", expected, "ancillas", got_ancillas, status)
print("all 16 input combinations checked")

all 16 input combinations checked


## 回路を `Circuit` オブジェクトではなくデータとして表現する

回路の*すべての*ゲートに汎用的に制御を追加するには、まず回路を単純な `(controls, target)` のペアのリストとして記述し(「`controls` に含まれるすべての量子ビットが1のときだけ `target` を反転する」という意味です)、最後に(`mcx` を使って)実際の `Circuit` に変換するのが簡単です。回路全体に追加の制御を加えるには、コンパイルする前にそれをすべてのタプルの `controls` に追加するだけで済みます。これは第1回・第2回と同じcarry/sum/剰余加算器のロジックを、別の形で表現し直したものです。

In [4]:
def carry_ops(c, a, b, c_next):
    return [((a, b), c_next), ((a,), b), ((c, b), c_next)]

def carry_inv_ops(c, a, b, c_next):
    return [((c, b), c_next), ((a,), b), ((a, b), c_next)]

def sum_ops(c, a, b):
    return [((a,), b), ((c,), b)]

def add_ops(c, a, b, n):
    """b := a + b (carry register c, b has n+1 qubits). Same construction as Part 1."""
    ops = []
    for i in range(n - 1):
        ops += carry_ops(c[i], a[i], b[i], c[i + 1])
    ops += carry_ops(c[n - 1], a[n - 1], b[n - 1], b[n])
    ops.append(((a[n - 1],), b[n - 1]))
    ops += sum_ops(c[n - 1], a[n - 1], b[n - 1])
    for i in range(n - 2, -1, -1):
        ops += carry_inv_ops(c[i], a[i], b[i], c[i + 1])
        ops += sum_ops(c[i], a[i], b[i])
    return ops

def sub_ops(c, a, b, n):
    """b := b - a: add_ops's gate list, reversed."""
    return list(reversed(add_ops(c, a, b, n)))

def modulo_add_ops(c, a, b, N, t, n):
    """b := (a + b) mod N, as Part 2 -- same five steps, expressed as (controls, target) pairs.
    Native controls only; extra controls (for making the whole thing itself controlled) are added by compile_ops."""
    ops = []
    ops += add_ops(c, a, b, n)                                          # step 1: b += a
    ops += sub_ops(c, N, b, n)                                          # step 2: b -= N
    ops.append(((b[n],), t))                                            # step 3: copy overflow into t
    ops += [(ctrl + (t,), tgt) for ctrl, tgt in add_ops(c, N, b, n)]     # step 4: if t: b += N
    ops += sub_ops(c, a, b, n)                                          # step 5: cleanup...
    ops.append(((), b[n]))
    ops.append(((b[n],), t))
    ops.append(((), b[n]))
    ops += add_ops(c, a, b, n)
    return ops

def compile_ops(ops, extra_controls, ancillas):
    """Turn a (controls, target) list into an actual Circuit, with `extra_controls` appended to every gate."""
    circ = Circuit()
    for controls, target in ops:
        circ += mcx(list(controls) + list(extra_controls), target, ancillas)
    return circ

この作り直しが、第2回の(制御なしの)剰余加算器を正確に再現できることを、簡単な回帰テストで確認しておきます。

In [5]:
def run_modulo_adder(val_a, val_b, val_N, n, n_ancilla=3):
    c = list(range(n))
    a = list(range(n, 2 * n))
    b = list(range(2 * n, 3 * n + 1))
    N = list(range(3 * n + 1, 4 * n + 1))
    t = 4 * n + 1
    ancillas = list(range(4 * n + 2, 4 * n + 2 + n_ancilla))
    circ = Circuit(4 * n + 2 + n_ancilla)
    for i in range(n):
        if (val_a >> i) & 1: circ.x[a[i]]
        if (val_b >> i) & 1: circ.x[b[i]]
        if (val_N >> i) & 1: circ.x[N[i]]
    circ += compile_ops(modulo_add_ops(c, a, b, N, t, n), [], ancillas)
    total = circ.n_qubits
    state = circ.m[:].run(shots=1).most_common(1)[0][0]
    bits = [state[total - 1 - q] for q in b]
    return int("".join(reversed(bits)), 2)

for x, y, N in [(7, 6, 11), (14, 13, 15), (3, 5, 9)]:
    result = run_modulo_adder(x, y, N, 4)
    print(f"({x}+{y}) mod {N} = {result}  (expected {(x + y) % N}, {'OK' if result == (x + y) % N else 'MISMATCH'})")

(7+6) mod 11 = 2  (expected 2, OK)


(14+13) mod 15 = 12  (expected 12, OK)


(3+5) mod 9 = 8  (expected 8, OK)


## 制御乗算器

$x$ の各ビット $i$ について: まず古典定数 $(a \cdot 2^i) \bmod N$ を、単純な `X` ゲートで小さなスクラッチレジスタに読み込みます(これは回路構築時に分かっている情報です — $a$、$N$、$i$ はすべて既知の定数であり、重ね合わせに依存するものではありません)。次に、乗算器自身の制御量子ビットと$x_i$の*両方*で制御して `modulo_add_ops` を実行し、最後に `X` ゲートを元に戻してスクラッチレジスタを次のビットのために $\ket 0$ にリセットします。

In [6]:
def build_controlled_multiplier(mult_a, val_x, val_N, ctrl_value, n, n_ancilla=4):
    c = list(range(n))                                # carry register
    const = list(range(n, 2 * n))                      # scratch: holds (a * 2**i) mod N
    b = list(range(2 * n, 3 * n + 1))                  # result register (n+1 wide)
    N = list(range(3 * n + 1, 4 * n + 1))               # modulus
    x = list(range(4 * n + 1, 5 * n + 1))               # multiplicand
    ctrl = 5 * n + 1                                    # multiplier's control qubit
    t = 5 * n + 2                                       # modulo-adder's overflow flag
    ancillas = list(range(5 * n + 3, 5 * n + 3 + n_ancilla))
    circ = Circuit(5 * n + 3 + n_ancilla)

    if ctrl_value:
        circ.x[ctrl]
    for i in range(n):
        if (val_x >> i) & 1:
            circ.x[x[i]]
        if (val_N >> i) & 1:
            circ.x[N[i]]

    for i in range(n):
        const_val = (mult_a * (1 << i)) % val_N
        for j in range(n):
            if (const_val >> j) & 1:
                circ.x[const[j]]
        circ += compile_ops(modulo_add_ops(c, const, b, N, t, n), [ctrl, x[i]], ancillas)
        for j in range(n):
            if (const_val >> j) & 1:
                circ.x[const[j]]

    return circ, b

In [7]:
def run_controlled_multiplier(mult_a, val_x, val_N, ctrl_value, n=None):
    """a, x, N, ctrl can be any values you like -- n (register width) is inferred from N and x
    automatically if not given explicitly."""
    if n is None:
        n = max(val_N, val_x, 1).bit_length()
    circuit, b = build_controlled_multiplier(mult_a, val_x, val_N, ctrl_value, n)
    total = circuit.n_qubits
    print(f"  [n={n}, qubits={total}, gates={len(circuit.ops)}]")
    state = circuit.m[:].run(shots=1).most_common(1)[0][0]
    bits = [state[total - 1 - q] for q in b]
    return int("".join(reversed(bits)), 2)

# n=2 (N up to 3) keeps this well within what this direct simulation approach can handle quickly.
for a, x, N, ctrl in [(2, 1, 3, 1), (2, 2, 3, 1), (2, 0, 3, 1), (2, 1, 3, 0)]:
    result = run_controlled_multiplier(a, x, N, ctrl)
    expected = (a * x) % N if ctrl else 0
    print(f"ctrl={ctrl}: {a}*{x} mod {N} = {result}  (expected {expected}, {'OK' if result == expected else 'MISMATCH'})")

  [n=2, qubits=17, gates=620]


ctrl=1: 2*1 mod 3 = 2  (expected 2, OK)
  [n=2, qubits=17, gates=620]


ctrl=1: 2*2 mod 3 = 1  (expected 1, OK)
  [n=2, qubits=17, gates=619]


ctrl=1: 2*0 mod 3 = 0  (expected 0, OK)
  [n=2, qubits=17, gates=619]


ctrl=0: 2*1 mod 3 = 0  (expected 0, OK)


## もう少し大きい例を試す

`run_controlled_multiplier` は任意の `a`、`x`、`N` を受け取れるので、上記のコードは特に $N=3$ に限定されているわけではありません — たまたま数秒で終わるサイズだったというだけです。この回路は(構築する際に量子的な重ね合わせを必要としない)普通のPythonコードだけで組み立てられるため、実行する前に量子ビット数とゲート数を正確に表示できます。ここでは少しだけ規模を上げて $N=7$ を実際に実行してみます: これは目に見えて時間がかかります(このノートブックを書いた環境ではおよそ2分)。これは $N=7$ がより多くの量子ビットを必要とするからではなく、ゲート数が急速に増えるためです。

In [8]:
import time

t0 = time.time()
result = run_controlled_multiplier(3, 5, 7, 1)  # 3 * 5 mod 7, controlled bit on
print(f"3*5 mod 7 = {result}  (expected {(3 * 5) % 7}, {'OK' if result == (3 * 5) % 7 else 'MISMATCH'})"
      f"  [{time.time() - t0:.0f}s]")

  [n=3, qubits=22, gates=1464]


3*5 mod 7 = 1  (expected 1, OK)  [149s]


4つのケースすべてが確認できました。特に重要なのは否定的なケースです: `ctrl=0` のとき、$x$ の値に関わらず結果レジスタは正確に `0` のままです — これにより、乗算全体が(最後のステップだけでなく)本当に条件付きであることが確認できます。

## 規模についての補足

上記の $N=7$ の例は、わずか22量子ビットの回路にもかかわらず、すでに数分かかりました。理由は量子ビット数ではなく**ゲート数**です: この回路は $x$ のビットごとに5ステップの剰余加算器全体を繰り返し、その中のすべてのゲートにさらに2つの制御が追加され、(合計3個以上の制御になる場合)それぞれが `mcx` を通じて余分な `CCX` ゲートを何個も必要とするからです。量子ビット数は $n$ に対して*線形*にしか増えませんが、このシミュレータの1ショットあたりのコストは、それらすべてのゲートを順番に適用しなければならないため、はるかに速く増大します。

また、単に遅いだけでなく構造的な壁もあります: このSDKのショット測定バックエンドは $2^{24}$ 通りの結果までしか扱えず、$N=15$(ショアのアルゴリズムの古典的な最初の例で、$n=4$ が必要)はここではすでに27量子ビットを必要とします — つまり $2^{27}$ 個の基底状態となり、その上限を超えてしまいます。$N=15$ に到達するには、より賢いシミュレーション戦略(例えば状態ベクトル全体を展開しない方法)、あるいはもちろん実際の量子コンピュータが必要です。実機であれば、回路を実行するコストは*シミュレータの*制約とはまったく関係がありません。この差 — 回路を組み立てるのは簡単なのに、それが大きくなるにつれて古典的にシミュレートするのがどんどん難しくなる — こそが、素因数分解がそもそも古典的に難しいと考えられている理由を、手を動かしながら少しだけ体感できる部分です。

## 次回予告

制御乗算ができたので、次の部品は[べき剰余](ja/404_shor_scratch_modular_exponentiation.ipynb)です — この回路を核として、位相推定レジスタの各ビットで制御しながら繰り返し二乗していきます。これがショアのアルゴリズムの量子部分が実際に実行する演算であり、このシリーズ全体を[400_shor.ipynb](ja/400_shor.ipynb)の位数発見の理論と最終的に結びつける部品です。